# 3. Agent, model, and tool middleware with recovery

## How this lesson maps to the live product

The flagship uses bounded sparse+dense fusion/reranking and policy-based retrieval critique/rewrite to feed OpenAI LLM `write_todos` planning. The Deep Agents supervisor delegates sequentially to recall-intelligence, product-lot-matching, traceability-reconciliation, then containment-communications. Each receives case-bound context and validated prerequisite claims; containment has no MCP tools. Safe projection yields safe typed claims, then the independent source verifier re-reads evidence before the HITL control plane can propose an action. LangGraph owns persistence, approval, execution confirmation and simulated receipts.

These embedded exercises teach individual mechanisms with deterministic local data. No live model was called. Their plans and metrics are teaching fixtures, not observed LLM output. Use `make ui-openai` with a private ignored .env for the live product and `make eval-model` for the additional measured lane. Live metrics remain unavailable until actually measured; offline tests/notebooks never spend API tokens. Semantic failures stop before review without fallback; provider/transport/budget failures may discard partial claims and take an explicitly labelled deterministic fallback. Raw prompts, model prose and chain-of-thought never belong in reports or checkpoints.


In [ ]:
print("Embedded dataset and deterministic offline lesson are ready.")

EDUCATIONAL — SELF-CONTAINED

Middleware is executable policy around an agent/model/tool boundary. This lesson records hook order, retries a transient model failure, masks a customer-like field, and blocks a write tool for a read-only role. The recovery path is deterministic and local.

In [ ]:
events: list[str] = []


def with_agent_context(agent, context):
    def wrapped(task):
        events.append("agent.before")
        result = agent(task, context)
        events.append("agent.after")
        return result
    return wrapped


def with_retry(model, attempts=3):
    def wrapped(prompt):
        for attempt in range(1, attempts + 1):
            events.append(f"model.attempt.{attempt}")
            try:
                return model(prompt)
            except RuntimeError:
                if attempt == attempts:
                    raise
                events.append("model.recover")
        raise AssertionError("unreachable")
    return wrapped


def with_tool_policy(tool, role):
    def wrapped(arguments):
        events.append("tool.before")
        if role == "reader" and tool.write_sensitive:
            raise PermissionError("reader cannot use write-sensitive tool")
        result = tool(arguments)
        events.append("tool.after")
        return result
    return wrapped


model_calls = 0

def flaky_model(prompt):
    global model_calls
    model_calls += 1
    if model_calls == 1:
        raise RuntimeError("transient model timeout")
    return {"answer": "inspect lot L-157"}


def read_tool(arguments):
    return {"lot": arguments["lot"], "customer": "MASKED"}


def write_tool(arguments):
    return {"receipt": "simulated"}

read_tool.write_sensitive = False
write_tool.write_sensitive = True
model = with_retry(flaky_model)
read = with_tool_policy(read_tool, "reader")
write = with_tool_policy(write_tool, "reader")

def agent(task, context):
    answer = model(task)
    observation = read({"lot": context["lot"]})
    return {**answer, **observation}

run = with_agent_context(agent, {"lot": "L-157"})("find affected lot")
print("recovered agent result ->", run)
print("middleware trace / observability events ->", events)
assert run["customer"] == "MASKED"
assert "model.recover" in events and model_calls == 2
try:
    write({"lot": "L-157"})
except PermissionError as error:
    print("permission recovery ->", error)
else:
    raise AssertionError("read-only agent reached a write tool")
print("ASSERTION PASSED: retry, masking, ordering, and permission policy worked")


In [ ]:
print("EDUCATIONAL — SELF-CONTAINED")
print("Recovery preserves the task while policy keeps unsafe capabilities out of reach.")
assert events[0] == "agent.before" and events[-1] == "tool.before"
print("ASSERTION PASSED: EDUCATIONAL — SELF-CONTAINED")